# Talent, Position Weights, And Economic Value

This notebook explores whether the economic target can be approximated as:

```text
economic_value ~= talent_score * position_weight
```

The goal is not to choose a final model here. The goal is to quickly test whether a talent-first model, such as one predicting `Peak_Overall`, plus learned positional conversion weights can explain much of the economic target used for draft-board decisions.

Primary questions:

- How strongly does `Peak_Overall` explain positive economic value?
- Do positions have meaningfully different talent-to-economic-value conversion rates?
- Does a simple `talent * position_weight` score produce useful draft-class rankings?
- Where does the talent-weighted board diverge from the raw economic target?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## Load Processed Data

This uses the current materialized feature store. The notebook expects to be run from the repository root or from the `notebooks/` directory.

In [ ]:
repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

features_path = repo_root / "fof8-ml" / "data" / "processed" / "features.parquet"
features_path

In [ ]:
raw = pl.read_parquet(features_path)

required = [
    "Player_ID",
    "Year",
    "Position_Group",
    "Position",
    "Peak_Overall",
    "Career_Merit_Cap_Share",
    "DPO",
    "Career_Games_Played",
]
available = [c for c in required if c in raw.columns]

df = raw.select(available).to_pandas()
df["Position_Group"] = df["Position_Group"].astype(str)
df["positive_econ"] = df["Career_Merit_Cap_Share"].clip(lower=0)
df["positive_dpo"] = df["DPO"].clip(lower=0)
df["log_positive_econ"] = np.log1p(df["positive_econ"])
df["is_positive_econ"] = df["positive_econ"] > 0

df.head()

In [ ]:
df[["Peak_Overall", "Career_Merit_Cap_Share", "positive_econ", "DPO", "Career_Games_Played"]].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])

## Position-Level Summary

This table gives a quick read on which positions convert observed peak talent into economic value at higher rates.

In [ ]:
position_summary = (
    df.groupby("Position_Group", observed=True)
    .agg(
        n=("Player_ID", "size"),
        positive_rate=("is_positive_econ", "mean"),
        mean_peak=("Peak_Overall", "mean"),
        mean_econ=("positive_econ", "mean"),
        p95_econ=("positive_econ", lambda s: s.quantile(0.95)),
        mean_games=("Career_Games_Played", "mean"),
    )
    .reset_index()
)
position_summary["econ_per_peak"] = position_summary["mean_econ"] / position_summary["mean_peak"].replace(0, np.nan)
position_summary.sort_values("mean_econ", ascending=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(
    data=position_summary.sort_values("mean_econ", ascending=False),
    x="Position_Group",
    y="mean_econ",
    ax=axes[0],
)
axes[0].set_title("Mean Positive Economic Value By Position")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=45)

sns.barplot(
    data=position_summary.sort_values("econ_per_peak", ascending=False),
    x="Position_Group",
    y="econ_per_peak",
    ax=axes[1],
)
axes[1].set_title("Mean Economic Value Per Peak Overall Point")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()

## Talent vs Economic Value

Use `log1p` for visualization because the economic target is heavily right-skewed.

In [ ]:
plot_df = df.sample(min(len(df), 25_000), random_state=42)

plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=plot_df,
    x="Peak_Overall",
    y="log_positive_econ",
    hue="Position_Group",
    alpha=0.35,
    s=18,
    linewidth=0,
)
plt.title("Peak Overall vs log1p(Positive Career Merit Cap Share)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", title="Position")
plt.tight_layout()

In [ ]:
plot_data = df[df["is_positive_econ"]].copy()

# Compute per-position linear R2 values for facet titles.
facet_r2 = {}
for pos, group in plot_data.groupby("Position_Group", observed=True):
    if len(group) < 2:
        facet_r2[pos] = np.nan
        continue
    model = LinearRegression().fit(group[["Peak_Overall"]], group["log_positive_econ"])
    facet_r2[pos] = model.score(group[["Peak_Overall"]], group["log_positive_econ"])

g = sns.lmplot(
    data=plot_data,
    x="Peak_Overall",
    y="log_positive_econ",
    col="Position_Group",
    col_wrap=4,
    scatter_kws={"alpha": 0.2, "s": 10},
    line_kws={"color": "black"},
    height=3,
)

for ax in g.axes.flat:
    pos = ax.get_title().split(" = ")[-1]
    r2 = facet_r2.get(pos)
    r2_label = "NA" if r2 is None or np.isnan(r2) else f"{r2:.2f}"
    ax.set_title(f"{pos} | R2={r2_label}")

g.fig.suptitle("Position-Specific Talent To Economic Conversion", y=1.02)


## Simple Baseline Regressions

These baselines test whether economic value is mostly talent, position, or talent-by-position interaction.

Interpretation:

- `peak_only`: a universal talent-to-economic conversion.
- `pos_only`: position priors without talent.
- `peak_plus_pos`: one universal talent slope plus position intercepts.
- `peak_x_pos`: position-specific talent slopes through zero.
- `peak_plus_pos_plus_interaction`: position intercepts plus position-specific slopes.

In [ ]:
def design_matrix(data: pd.DataFrame, mode: str, position_columns=None):
    if position_columns is None:
        position_columns = pd.get_dummies(data["Position_Group"]).columns

    peak = data[["Peak_Overall"]].to_numpy()
    pos = pd.get_dummies(data["Position_Group"]).reindex(columns=position_columns, fill_value=0).to_numpy()

    if mode == "peak_only":
        return peak, position_columns
    if mode == "pos_only":
        return pos, position_columns
    if mode == "peak_plus_pos":
        return np.column_stack([peak, pos]), position_columns
    if mode == "peak_x_pos":
        return pos * peak, position_columns
    if mode == "peak_plus_pos_plus_interaction":
        return np.column_stack([peak, pos, pos * peak]), position_columns

    raise ValueError(f"Unknown mode: {mode}")


def evaluate_baselines(data: pd.DataFrame, target_col: str):
    modes = [
        "peak_only",
        "pos_only",
        "peak_plus_pos",
        "peak_x_pos",
        "peak_plus_pos_plus_interaction",
    ]
    rows = []
    position_columns = pd.get_dummies(data["Position_Group"]).columns
    y = data[target_col].to_numpy()

    for mode in modes:
        X, _ = design_matrix(data, mode, position_columns)
        model = LinearRegression().fit(X, y)
        pred = model.predict(X)
        rows.append(
            {
                "target": target_col,
                "subset_n": len(data),
                "model": mode,
                "r2": r2_score(y, pred),
                "mae": mean_absolute_error(y, pred),
            }
        )

    return pd.DataFrame(rows)

In [ ]:
baseline_results = pd.concat(
    [
        evaluate_baselines(df, "positive_econ").assign(subset="all"),
        evaluate_baselines(df, "log_positive_econ").assign(subset="all"),
        evaluate_baselines(df[df["is_positive_econ"]], "positive_econ").assign(subset="positive_only"),
        evaluate_baselines(df[df["is_positive_econ"]], "log_positive_econ").assign(subset="positive_only"),
    ],
    ignore_index=True,
)

baseline_results.sort_values(["subset", "target", "r2"], ascending=[True, True, False])

## Learn Position Conversion Weights

Fit a separate simple line per position:

```text
log1p(positive_economic_value) ~= intercept_position + slope_position * Peak_Overall
```

The slope is a rough estimate of how much economic value each point of peak talent converts into for that position.

In [ ]:
def fit_position_slopes(data: pd.DataFrame):
    rows = []
    for pos, group in data.groupby("Position_Group", observed=True):
        if len(group) < 20:
            continue
        X = group[["Peak_Overall"]].to_numpy()
        y = group["log_positive_econ"].to_numpy()
        model = LinearRegression().fit(X, y)
        rows.append(
            {
                "Position_Group": pos,
                "n": len(group),
                "intercept": model.intercept_,
                "slope": model.coef_[0],
                "r2": model.score(X, y),
                "mean_peak": group["Peak_Overall"].mean(),
                "mean_positive_econ": group["positive_econ"].mean(),
            }
        )
    return pd.DataFrame(rows).sort_values("slope", ascending=False)


position_slopes = fit_position_slopes(df[df["is_positive_econ"]])
position_slopes

In [ ]:
plt.figure(figsize=(12, 5))
sns.barplot(data=position_slopes, x="Position_Group", y="slope")
plt.title("Position-Specific log Economic Value Per Peak Overall Point")
plt.xlabel("")
plt.xticks(rotation=45)
plt.tight_layout()

## Residual Check: What Does Talent-Only Miss?

If a universal talent model is missing positional economics, residuals should show systematic position effects.

In [ ]:
talent_model = LinearRegression().fit(df[["Peak_Overall"]], df["log_positive_econ"])
df_resid = df.copy()
df_resid["talent_only_log_pred"] = talent_model.predict(df[["Peak_Overall"]])
df_resid["talent_only_residual"] = df_resid["log_positive_econ"] - df_resid["talent_only_log_pred"]

resid_summary = (
    df_resid.groupby("Position_Group", observed=True)
    .agg(
        n=("Player_ID", "size"),
        mean_residual=("talent_only_residual", "mean"),
        median_residual=("talent_only_residual", "median"),
    )
    .reset_index()
    .sort_values("mean_residual", ascending=False)
)
resid_summary

In [ ]:
plt.figure(figsize=(12, 5))
sns.barplot(data=resid_summary, x="Position_Group", y="mean_residual")
plt.axhline(0, color="black", linewidth=1)
plt.title("Mean Residual By Position After Talent-Only Economic Model")
plt.xlabel("")
plt.xticks(rotation=45)
plt.tight_layout()

## Draft-Class Ranking Metrics

These quick metrics apply `K` within each draft class, then average across draft classes. This matches the real decision context: ranking one live draft class at a time.

In [ ]:
def ndcg_at_k(y_true, y_score, k):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    order = np.argsort(y_score)[::-1][:k]
    gains = np.maximum(y_true[order], 0)
    discounts = 1 / np.log2(np.arange(2, len(gains) + 2))
    dcg = np.sum(gains * discounts)
    ideal = np.sort(np.maximum(y_true, 0))[::-1][:k]
    ideal_dcg = np.sum(ideal * discounts)
    return 0.0 if ideal_dcg == 0 else float(dcg / ideal_dcg)


def mean_ndcg_by_year(data, score_col, target_col, k):
    scores = []
    for _, group in data.groupby("Year", observed=True):
        scores.append(ndcg_at_k(group[target_col].to_numpy(), group[score_col].to_numpy(), k=k))
    return float(np.mean(scores))


def topk_actual_value_by_year(data, score_col, target_col, k):
    values = []
    for _, group in data.groupby("Year", observed=True):
        top = group.sort_values(score_col, ascending=False).head(k)
        values.append(top[target_col].clip(lower=0).sum())
    return float(np.mean(values))


def bust_rate_by_year(data, score_col, success_col, k):
    rates = []
    for _, group in data.groupby("Year", observed=True):
        top = group.sort_values(score_col, ascending=False).head(k)
        rates.append(1 - top[success_col].mean())
    return float(np.mean(rates))

## Compare Simple Board Scores

This compares a talent-only score, an economic target baseline, and simple position-weighted talent scores. The `actual_economic` score is an oracle upper-bound reference, not a usable model.

In [ ]:
score_df = df.copy()

# Mean-value weights: simple average economic value per peak point by position.
weight_map = position_summary.set_index("Position_Group")["econ_per_peak"].to_dict()
score_df["score_peak_only"] = score_df["Peak_Overall"]
score_df["score_weighted_peak_mean"] = score_df["Peak_Overall"] * score_df["Position_Group"].map(weight_map)

# Slope weights: position-specific log-economic slope among positive players.
slope_map = position_slopes.set_index("Position_Group")["slope"].to_dict()
score_df["score_weighted_peak_slope"] = score_df["Peak_Overall"] * score_df["Position_Group"].map(slope_map)

# Oracle target reference.
score_df["score_actual_economic_oracle"] = score_df["positive_econ"]

score_cols = [
    "score_peak_only",
    "score_weighted_peak_mean",
    "score_weighted_peak_slope",
    "score_actual_economic_oracle",
]

rows = []
for score_col in score_cols:
    rows.append(
        {
            "score": score_col,
            "econ_ndcg_at_32": mean_ndcg_by_year(score_df, score_col, "positive_econ", 32),
            "econ_ndcg_at_64": mean_ndcg_by_year(score_df, score_col, "positive_econ", 64),
            "peak_ndcg_at_64": mean_ndcg_by_year(score_df, score_col, "Peak_Overall", 64),
            "games_ndcg_at_64": mean_ndcg_by_year(score_df, score_col, "Career_Games_Played", 64),
            "top64_actual_econ": topk_actual_value_by_year(score_df, score_col, "positive_econ", 64),
            "bust_rate_at_32": bust_rate_by_year(score_df, score_col, "is_positive_econ", 32),
        }
    )

score_comparison = pd.DataFrame(rows).sort_values("econ_ndcg_at_64", ascending=False)
score_comparison

## Inspect Divergences

The most useful rows to inspect are players that talent-only and position-weighted talent rank very differently.

In [ ]:
inspect_year = int(score_df["Year"].max())
year_board = score_df[score_df["Year"] == inspect_year].copy()

for col in ["score_peak_only", "score_weighted_peak_mean", "score_weighted_peak_slope", "positive_econ"]:
    year_board[f"rank_{col}"] = year_board[col].rank(ascending=False, method="first")

year_board["weighted_vs_peak_rank_delta"] = year_board["rank_score_peak_only"] - year_board["rank_score_weighted_peak_mean"]

cols = [
    "Player_ID",
    "Year",
    "Position_Group",
    "Position",
    "Peak_Overall",
    "positive_econ",
    "Career_Games_Played",
    "score_peak_only",
    "score_weighted_peak_mean",
    "rank_score_peak_only",
    "rank_score_weighted_peak_mean",
    "weighted_vs_peak_rank_delta",
]

year_board.sort_values("weighted_vs_peak_rank_delta", ascending=False)[cols].head(25)

In [ ]:
year_board.sort_values("weighted_vs_peak_rank_delta", ascending=True)[cols].head(25)

## Next Steps

If these simple weighted-talent scores look promising:

1. Replace `Peak_Overall` with out-of-fold predictions from a talent regressor.
2. Learn position weights only on the training years, then evaluate on held-out draft years.
3. Compare against a direct economic regressor with the same per-draft-class cross-outcome metrics.
4. Inspect top-board divergences manually by draft class.

The strongest evidence for the hybrid approach would be that `predicted_talent * learned_position_weight` preserves most economic NDCG/top-k value while improving talent/longevity robustness and interpretability.

## Chronological Hold-Out Analysis (Train Weights, Test Board)

This section avoids in-sample leakage: position conversion weights are learned on older draft years only, then evaluated on later years.


In [ ]:
# Split point can be adjusted; keep enough years on each side.
split_year = int(df['Year'].quantile(0.8))

train_df = df[df['Year'] <= split_year].copy()
test_df = df[df['Year'] > split_year].copy()

print('split_year:', split_year)
print('train years:', int(train_df['Year'].min()), 'to', int(train_df['Year'].max()), 'n=', len(train_df))
print('test years :', int(test_df['Year'].min()), 'to', int(test_df['Year'].max()), 'n=', len(test_df))


In [ ]:
# Learn slope weights only from positive-economic examples in training years.
train_pos = train_df[train_df['positive_econ'] > 0].copy()

slope_rows = []
for pos, g in train_pos.groupby('Position_Group', observed=True):
    if len(g) < 20:
        continue
    X = g[['Peak_Overall']].to_numpy()
    y = np.log1p(g['positive_econ'].to_numpy())
    m = LinearRegression().fit(X, y)
    slope_rows.append({'Position_Group': pos, 'slope': float(m.coef_[0]), 'intercept': float(m.intercept_), 'n': len(g)})

train_slopes = pd.DataFrame(slope_rows).sort_values('slope', ascending=False)
train_slope_map = train_slopes.set_index('Position_Group')['slope'].to_dict()
global_fallback_slope = float(train_slopes['slope'].median())

train_slopes


In [ ]:
# Build scores for hold-out test years only.
holdout = test_df.copy()
holdout['score_peak_only'] = holdout['Peak_Overall']
holdout['position_slope_weight'] = holdout['Position_Group'].map(train_slope_map).fillna(global_fallback_slope)
holdout['score_weighted_peak_slope_holdout'] = holdout['Peak_Overall'] * holdout['position_slope_weight']

holdout[['Position_Group', 'Peak_Overall', 'position_slope_weight', 'score_peak_only', 'score_weighted_peak_slope_holdout']].head()


In [ ]:
# Compare hold-out ranking and value capture.
rows = []
for score_col in ['score_peak_only', 'score_weighted_peak_slope_holdout']:
    rows.append({
        'score': score_col,
        'econ_ndcg_at_32': mean_ndcg_by_year(holdout, score_col, 'positive_econ', 32),
        'econ_ndcg_at_64': mean_ndcg_by_year(holdout, score_col, 'positive_econ', 64),
        'peak_ndcg_at_64': mean_ndcg_by_year(holdout, score_col, 'Peak_Overall', 64),
        'games_ndcg_at_64': mean_ndcg_by_year(holdout, score_col, 'Career_Games_Played', 64),
        'top64_actual_econ': topk_actual_value_by_year(holdout, score_col, 'positive_econ', 64),
        'bust_rate_at_32': bust_rate_by_year(holdout, score_col, 'is_positive_econ', 32),
    })

holdout_comparison = pd.DataFrame(rows).sort_values('econ_ndcg_at_64', ascending=False)
holdout_comparison


In [ ]:
# Year-by-year difference view: where weighted slope helps or hurts.
year_rows = []
for y, g in holdout.groupby('Year', observed=True):
    peak_ndcg = ndcg_at_k(g['positive_econ'].to_numpy(), g['score_peak_only'].to_numpy(), 64)
    w_ndcg = ndcg_at_k(g['positive_econ'].to_numpy(), g['score_weighted_peak_slope_holdout'].to_numpy(), 64)

    top_peak = g.sort_values('score_peak_only', ascending=False).head(64)['positive_econ'].clip(lower=0).sum()
    top_w = g.sort_values('score_weighted_peak_slope_holdout', ascending=False).head(64)['positive_econ'].clip(lower=0).sum()

    year_rows.append({
        'Year': int(y),
        'econ_ndcg64_peak_only': peak_ndcg,
        'econ_ndcg64_weighted': w_ndcg,
        'delta_ndcg64_weighted_minus_peak': w_ndcg - peak_ndcg,
        'top64_econ_peak_only': float(top_peak),
        'top64_econ_weighted': float(top_w),
        'delta_top64_econ_weighted_minus_peak': float(top_w - top_peak),
    })

year_deltas = pd.DataFrame(year_rows).sort_values('Year')
year_deltas


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=year_deltas, x='Year', y='delta_ndcg64_weighted_minus_peak', ax=axes[0], color='steelblue')
axes[0].axhline(0, color='black', linewidth=1)
axes[0].set_title('Hold-Out Year Delta: Econ NDCG@64 (Weighted - Peak)')
axes[0].tick_params(axis='x', rotation=45)

sns.barplot(data=year_deltas, x='Year', y='delta_top64_econ_weighted_minus_peak', ax=axes[1], color='darkorange')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_title('Hold-Out Year Delta: Top64 Actual Econ (Weighted - Peak)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()


### Optional: Fixed Split Sensitivity

Repeat the hold-out check with different split years (for example 60/40, 70/30, 80/20) to ensure the conclusion is not a split artifact.


In [ ]:
def evaluate_split(split_year):
    tr = df[df['Year'] <= split_year].copy()
    te = df[df['Year'] > split_year].copy()

    tr_pos = tr[tr['positive_econ'] > 0]
    rows = []
    for pos, g in tr_pos.groupby('Position_Group', observed=True):
        if len(g) < 20:
            continue
        X = g[['Peak_Overall']].to_numpy()
        y = np.log1p(g['positive_econ'].to_numpy())
        m = LinearRegression().fit(X, y)
        rows.append((pos, float(m.coef_[0])))

    if not rows or te.empty:
        return None

    slope_map = dict(rows)
    fallback = float(np.median([v for _, v in rows]))

    te = te.copy()
    te['score_peak_only'] = te['Peak_Overall']
    te['score_weighted'] = te['Peak_Overall'] * te['Position_Group'].map(slope_map).fillna(fallback)

    out = {
        'split_year': int(split_year),
        'n_train': int(len(tr)),
        'n_test': int(len(te)),
        'econ_ndcg64_peak': mean_ndcg_by_year(te, 'score_peak_only', 'positive_econ', 64),
        'econ_ndcg64_weighted': mean_ndcg_by_year(te, 'score_weighted', 'positive_econ', 64),
        'top64_econ_peak': topk_actual_value_by_year(te, 'score_peak_only', 'positive_econ', 64),
        'top64_econ_weighted': topk_actual_value_by_year(te, 'score_weighted', 'positive_econ', 64),
    }
    out['delta_ndcg64'] = out['econ_ndcg64_weighted'] - out['econ_ndcg64_peak']
    out['delta_top64_econ'] = out['top64_econ_weighted'] - out['top64_econ_peak']
    return out

split_candidates = sorted(df['Year'].unique())
split_candidates = [y for y in split_candidates if y > df['Year'].min() + 10 and y < df['Year'].max() - 5]
sample_splits = [split_candidates[int(len(split_candidates) * q)] for q in [0.6, 0.7, 0.8]]

sensitivity_rows = [evaluate_split(y) for y in sample_splits]
sensitivity_rows = [r for r in sensitivity_rows if r is not None]
pd.DataFrame(sensitivity_rows)
